In [137]:
import numpy as np
import pandas as pd
import scipy
import re
import os
from pathlib import Path

Raw datasets downloaded from https://www.elsevier.com/products/scopus/content

on 13 May 2026

(according to Scopus, last updated March 2026)

In [138]:
# current sources

current_df = pd.read_excel('../data/260513_scopus/ext_list_Mar_2026.xlsx', sheet_name='Scopus Sources Mar. 2026')

In [139]:
current_df['Active or Inactive'].value_counts()

Active      31570
Inactive    16790
Name: Active or Inactive, dtype: int64

In [140]:
current_df = current_df[current_df['Active or Inactive'] == 'Active'].copy()

In [141]:
# past sources

past_df = pd.read_excel('../data/260513_scopus/ext_list_Mar_2026.xlsx', 
                        sheet_name='Discontinued Titles Mar. 2026', header=1)

In [142]:
current_df['indexing_status'] = 'indexed'
current_df['journal_title'] = current_df['Source Title']
past_df['indexing_status'] = 'deindexed'
past_df['journal_title'] = past_df['Source Title (newly added titles are highlighted in red)']
past_df['notes_on_indexing_status'] = past_df['Indexation Change']

In [143]:
# conference proceedings

conf_df = pd.read_excel('../data/260513_scopus/ext_list_Mar_2026.xlsx', 
                        sheet_name='Serial Conf. Proc. with Profile', header=0)

def mapper(f):
    if str(f) == 'nan':
        return 'indexed'
    else:
        return 'deindexed'
    
conf_df['indexing_status'] = conf_df['Titles Discontinued by Scopus'].apply(mapper)
conf_df['journal_title'] = conf_df['Source Title']
conf_df['notes_on_indexing_status'] = 'serial conference proceedings'

In [144]:
df_all = pd.concat([current_df, past_df, conf_df])

In [145]:
def process_issn(issn):
    issn = str(issn)
    if issn != 'nan':
        return issn[0:4] + '-' + issn[4:]
    else:
        return np.nan

In [146]:
df_all['p_issn'] = df_all['ISSN'].apply(process_issn)
df_all['e_issn'] = df_all['EISSN'].apply(process_issn)
df_all['service'] = 'scopus'
df_all['internal_identifier'] = df_all['Sourcerecord ID']

In [147]:
df_all['edition'] = '260513'

In [148]:
df_all_slice = df_all[['service', 'edition', 'internal_identifier', 
                       'journal_title', 'p_issn', 'e_issn', 
                       'indexing_status', 'notes_on_indexing_status']].copy()

In [149]:
file_prefix = '260513_scopus'
df_all_slice.to_csv('../data/' + file_prefix + '.csv', index=False)
df_all_slice.to_parquet('../data/' + file_prefix + '.parquet')